### 1. Import Library
Modul `re` adalah pustaka bawaan Python untuk Regular Expression (Regex). Dalam pipeline NLP dan preprocessing dataset untuk large model, regex adalah senjata pertama sebelum tokenizer bahkan berjalan — membersihkan teks kotor dari URL, karakter khusus, format aneh, hingga ekstraksi pola spesifik.

In [1]:
import re

# Fungsi helper untuk demo bersih
def show(label, result):
    print(f"  [{label}] → {result!r}")

### 2. Sintaks Dasar Regex — Karakter & Metacharacter

Regex dibangun dari **karakter literal** dan **metacharacter** (karakter dengan makna khusus). Berikut adalah referensi lengkap yang perlu dikuasai:

---

#### 🔹 Karakter Literal
| Pattern | Arti | Contoh |
|---------|------|--------|
| `abc`   | Cocokkan huruf 'a', 'b', 'c' berurutan | `re.search(r'cat', 'the cat sat')` |
| `123`   | Cocokkan angka '1', '2', '3' berurutan | `re.search(r'123', 'abc123')` |

---

#### 🔹 Character Classes `[ ]`
| Pattern | Arti | Contoh cocok |
|---------|------|--------------|
| `[abc]` | Salah satu dari a, b, atau c | `'a'`, `'b'`, `'c'` |
| `[a-z]` | Huruf kecil a sampai z | `'m'`, `'x'` |
| `[A-Z]` | Huruf besar A sampai Z | `'M'`, `'X'` |
| `[0-9]` | Angka 0 sampai 9 | `'3'`, `'7'` |
| `[a-zA-Z0-9]` | Semua huruf dan angka | `'a'`, `'Z'`, `'5'` |
| `[^abc]` | Bukan a, b, atau c (negasi `^`) | `'d'`, `'1'`, `' '` |
| `[^0-9]` | Bukan angka | `'a'`, `'!'` |

---

#### 🔹 Shorthand Character Classes
| Pattern | Arti | Setara dengan |
|---------|------|---------------|
| `\d` | Digit (angka) | `[0-9]` |
| `\D` | Bukan digit | `[^0-9]` |
| `\w` | Word character (huruf, angka, underscore) | `[a-zA-Z0-9_]` |
| `\W` | Bukan word character | `[^a-zA-Z0-9_]` |
| `\s` | Whitespace (spasi, tab, newline) | `[ \t\n\r\f\v]` |
| `\S` | Bukan whitespace | `[^ \t\n\r\f\v]` |
| `.`  | Karakter apapun **kecuali** newline (default) | *(semua kecuali `\n`)* |

---

#### 🔹 Anchors (Posisi, Bukan Karakter)
| Pattern | Arti |
|---------|------|
| `^` | Awal string (atau awal baris jika flag `MULTILINE`) |
| `$` | Akhir string (atau akhir baris jika flag `MULTILINE`) |
| `\b` | Word boundary — batas antara `\w` dan `\W` |
| `\B` | Bukan word boundary |
| `\A` | Awal string (selalu, tidak terpengaruh `MULTILINE`) |
| `\Z` | Akhir string (selalu, tidak terpengaruh `MULTILINE`) |

---

#### 🔹 Quantifiers (Berapa Kali Berulang)
| Pattern | Arti | Contoh |
|---------|------|--------|
| `*`     | 0 atau lebih kali | `ab*` cocok `'a'`, `'ab'`, `'abbb'` |
| `+`     | 1 atau lebih kali | `ab+` cocok `'ab'`, `'abbb'` tapi **bukan** `'a'` |
| `?`     | 0 atau 1 kali (opsional) | `colou?r` cocok `'color'` dan `'colour'` |
| `{n}`   | Tepat n kali | `\d{4}` cocok tepat 4 digit |
| `{n,}`  | Minimal n kali | `\d{2,}` cocok 2 digit atau lebih |
| `{n,m}` | Antara n dan m kali | `\d{2,4}` cocok 2, 3, atau 4 digit |

> **Greedy vs Lazy:** Secara default quantifier bersifat *greedy* (ambil sebanyak mungkin). Tambahkan `?` setelah quantifier untuk *lazy* (ambil sesedikit mungkin): `*?`, `+?`, `{n,m}?`

---

#### 🔹 Groups dan Alternation
| Pattern | Arti |
|---------|------|
| `(abc)` | Capturing group — cocokkan dan **tangkap** `'abc'` |
| `(?:abc)` | Non-capturing group — cocokkan tapi **jangan tangkap** |
| `(?P<name>abc)` | Named capturing group — tangkap dengan nama `'name'` |
| `a\|b` | Alternation — cocokkan `'a'` **atau** `'b'` |
| `\1` | Backreference ke group ke-1 |

---

#### 🔹 Lookahead & Lookbehind (Zero-Width Assertions)
| Pattern | Arti |
|---------|------|
| `(?=...)` | Positive lookahead — diikuti oleh pola ini |
| `(?!...)` | Negative lookahead — **tidak** diikuti oleh pola ini |
| `(?<=...)` | Positive lookbehind — didahului oleh pola ini |
| `(?<!...)` | Negative lookbehind — **tidak** didahului oleh pola ini |

---

#### 🔹 Escape Character
| Pattern | Arti |
|---------|------|
| `\.` | Titik literal (bukan metacharacter) |
| `\*` | Bintang literal |
| `\(` `\)` | Kurung literal |
| `\\` | Backslash literal |

> **Tips Python:** Selalu gunakan **raw string** `r'...'` untuk pola regex agar backslash tidak perlu di-escape dua kali. Contoh: `r'\d+'` bukan `'\\d+'`

In [2]:
import re

# ================================================================
# DEMO INTERAKTIF — Sintaks Regex dari Dasar
# ================================================================

def demo(label, pattern, text, func='findall'):
    if func == 'findall':
        result = re.findall(pattern, text)
    elif func == 'search':
        m = re.search(pattern, text)
        result = m.group() if m else None
    print(f"  {label:<30} pattern={pattern!r:<28} → {result}")

sample = "Hello World! Model BERT v2.0 trained 3 epochs, loss=0.312, acc=94.5%"
print(f"Teks: {sample!r}\n")

# --- [1] Character Classes ---
print("=" * 70)
print("[1] Character Classes")
print("=" * 70)
demo('huruf kecil berurutan',    r'[a-z]+',      sample)
demo('huruf besar berurutan',    r'[A-Z]+',      sample)
demo('digit berurutan',          r'[0-9]+',      sample)
demo('hanya vokal',              r'[aeiou]',     sample)
demo('bukan huruf & spasi',      r'[^a-zA-Z ]',  sample)

# --- [2] Shorthand Character Classes ---
print("\n" + "=" * 70)
print("[2] Shorthand Character Classes")
print("=" * 70)
demo('\\d  — semua digit',        r'\d+',   sample)
demo('\\w  — semua kata',          r'\w+',   sample)
demo('\\s  — semua whitespace',    r'\s',    sample)
demo('\\S  — non-whitespace',      r'\S+',   sample)
demo('.   — karakter apapun',     r'.{5}',  sample)

# --- [3] Quantifiers ---
print("\n" + "=" * 70)
print("[3] Quantifiers")
print("=" * 70)
demo('{n,m} — 1 s/d 3 digit',    r'\d{1,3}',     sample)
demo('+     — angka + persen',    r'[\d.]+%',     sample)
demo('+     — versi v2.0',        r'v\d+\.\d+',  sample)
demo('?     — u opsional',        r'colou?r',     'I love color and colour')

# --- [4] Anchors ---
print("\n" + "=" * 70)
print("[4] Anchors")
print("=" * 70)
demo('^  — kata pertama',         r'^\w+',      sample, func='search')
demo('$  — kata terakhir',         r'[\d.]+%$',  sample, func='search')
demo('\\b — word boundary loss',   r'\bloss\b',  sample, func='search')

# --- [5] Greedy vs Lazy ---
print("\n" + "=" * 70)
print("[5] Greedy vs Lazy")
print("=" * 70)
tagged = "<b>Hello</b> and <i>World</i>"
print(f"  Teks: {tagged!r}")
demo('greedy <.*>',               r'<.*>',   tagged, func='search')
demo('lazy   <.*?>',              r'<.*?>',  tagged)

# --- [6] Groups, Alternation, Backreference ---
print("\n" + "=" * 70)
print("[6] Groups & Alternation")
print("=" * 70)
tech = "PyTorch and TensorFlow are frameworks. Keras is also popular."
demo('alternation |',             r'PyTorch|TensorFlow|Keras', tech)

m = re.search(r'loss=([\d.]+)', sample)
print(f"  {'capturing group (1)':30s} → {m.group(1)!r} (loss value)")

m = re.search(r'acc=(?P<accuracy>[\d.]+)', sample)
print(f"  {'named group accuracy':30s} → {m.group('accuracy')!r}")

# backreference: kata yang diulang
repeat_text = "the the model model is is good"
found_repeat = re.findall(r'\b(\w+)\s+\1\b', repeat_text)
print(f"  {'backreference (kata ulang)':30s} → {found_repeat}")

# --- [7] Lookahead & Lookbehind ---
print("\n" + "=" * 70)
print("[7] Lookahead & Lookbehind")
print("=" * 70)
demo('(?=%) angka sebelum %',     r'[\d.]+(?=%)',   sample)
demo('(?<=loss=) nilai loss',     r'(?<=loss=)[\d.]+', sample)
demo('(?!) bukan v+digit',        r'\b(?!v\d)\w{4,}\b', sample)

Teks: 'Hello World! Model BERT v2.0 trained 3 epochs, loss=0.312, acc=94.5%'

[1] Character Classes
  huruf kecil berurutan          pattern='[a-z]+'                     → ['ello', 'orld', 'odel', 'v', 'trained', 'epochs', 'loss', 'acc']
  huruf besar berurutan          pattern='[A-Z]+'                     → ['H', 'W', 'M', 'BERT']
  digit berurutan                pattern='[0-9]+'                     → ['2', '0', '3', '0', '312', '94', '5']
  hanya vokal                    pattern='[aeiou]'                    → ['e', 'o', 'o', 'o', 'e', 'a', 'i', 'e', 'e', 'o', 'o', 'a']
  bukan huruf & spasi            pattern='[^a-zA-Z ]'                 → ['!', '2', '.', '0', '3', ',', '=', '0', '.', '3', '1', '2', ',', '=', '9', '4', '.', '5', '%']

[2] Shorthand Character Classes
  \d  — semua digit              pattern='\\d+'                       → ['2', '0', '3', '0', '312', '94', '5']
  \w  — semua kata               pattern='\\w+'                       → ['Hello', 'World', 'Model', 'BERT', 'v

### 3. Cheatsheet Pola Regex untuk NLP & LLM Pipeline

Kumpulan pola siap pakai yang paling sering muncul di preprocessing dataset dan pipeline model:

| Tujuan | Pattern | Keterangan |
|--------|---------|------------|
| URL | `https?://\S+\|www\.\S+` | http dan https |
| Email | `[\w.+-]+@[\w-]+\.[a-z]{2,}` | Format email standar |
| Mention Twitter | `@[A-Za-z0-9_]+` | Username @mention |
| Hashtag | `#\w+` | Hashtag media sosial |
| Angka bulat | `\b\d+\b` | Integer |
| Angka desimal | `\b\d+\.\d+\b` | Float |
| Notasi ilmiah | `\d+(?:\.\d+)?[eE][+-]?\d+` | `5e-5`, `1.2E+3` |
| Persentase | `[\d.]+%` | Angka dengan `%` |
| HTML/XML tag | `<[^>]+>` | Semua tag |
| Non-ASCII/emoji | `[^\x00-\x7F]+` | Di luar ASCII |
| Spasi berlebih | `\s+` | Normalisasi whitespace |
| Tanda baca berulang | `([!?.])\1+` | `!!!!` → `!` |
| Kata berulang | `\b(\w+)(\s+\1){2,}\b` | `good good good` |
| Kode Python | `def \|import \|class \|return ` | Indikator ada kode |
| Versi software | `v?\d+\.\d+(?:\.\d+)?` | `v1.0`, `2.3.1` |
| Tanggal ISO | `\d{4}-\d{2}-\d{2}` | `2024-01-31` |
| Waktu | `\d{2}:\d{2}(?::\d{2})?` | `14:30`, `09:15:22` |
| Blok kode markdown | `` \`\`\`(\w+)?\n(.*?)\`\`\` `` | Dengan flag `DOTALL` |
| Section Alpaca | `### (Instruction\|Input\|Response):` | Format prompt LLM |

In [3]:
# ================================================================
# Demo Cheatsheet — uji semua pola sekaligus pada satu teks
# ================================================================

import re

test_text = (
    "On 2024-01-31 at 09:15:22, @JohnDoe emailed user@example.com saying: "
    "'Check https://huggingface.co/bert v4.35.2 — accuracy=94.5%, loss=5e-4!!! "
    "<b>Great</b> work  #AI #PyTorch'"
)

patterns = {
    'URL'               : r'https?://\S+|www\.\S+',
    'Email'             : r'[\w.+-]+@[\w-]+\.[a-z]{2,}',
    'Mention (@)'       : r'@[A-Za-z0-9_]+',
    'Hashtag (#)'       : r'#\w+',
    'Angka desimal'     : r'\b\d+\.\d+\b',
    'Notasi ilmiah'     : r'\d+(?:\.\d+)?[eE][+-]?\d+',
    'Persentase'        : r'[\d.]+(?=%)',
    'HTML tag'          : r'<[^>]+>',
    'Non-ASCII/emoji'   : r'[^\x00-\x7F]+',
    'Tanggal ISO'       : r'\d{4}-\d{2}-\d{2}',
    'Waktu'             : r'\d{2}:\d{2}(?::\d{2})?',
    'Versi software'    : r'v?\d+\.\d+(?:\.\d+)?',
    'Tanda baca ulang'  : r'([!?.])\1+',
}

print(f"Teks uji:\n  {test_text!r}\n")
print(f"  {'Pola':<22} {'Hasil'}")
print("  " + "-" * 65)
for name, pattern in patterns.items():
    found = re.findall(pattern, test_text)
    print(f"  {name:<22} {found}")

Teks uji:
  "On 2024-01-31 at 09:15:22, @JohnDoe emailed user@example.com saying: 'Check https://huggingface.co/bert v4.35.2 — accuracy=94.5%, loss=5e-4!!! <b>Great</b> work  #AI #PyTorch'"

  Pola                   Hasil
  -----------------------------------------------------------------
  URL                    ['https://huggingface.co/bert']
  Email                  ['user@example.com']
  Mention (@)            ['@JohnDoe', '@example']
  Hashtag (#)            ['#AI', '#PyTorch']
  Angka desimal          ['35.2', '94.5']
  Notasi ilmiah          ['5e-4']
  Persentase             ['94.5']
  HTML tag               ['<b>', '</b>']
  Non-ASCII/emoji        ['—']
  Tanggal ISO            ['2024-01-31']
  Waktu                  ['09:15:22']
  Versi software         ['v4.35.2', '94.5']
  Tanda baca ulang       ['!']


### 6. Fungsi Dasar: `re.search`, `re.match`, `re.fullmatch`, `re.findall`
Keempat fungsi ini adalah titik masuk paling sering digunakan. Dalam konteks preprocessing LLM, `re.findall` adalah yang paling banyak dipakai untuk mengekstrak semua kemunculan pola (misal semua URL, semua mention, semua angka) dari teks, sedangkan `re.search` digunakan untuk mengecek apakah suatu pola *ada* dalam teks.

In [4]:
text = "Check https://huggingface.co/bert-base and also http://arxiv.org/abs/1706.03762 for papers."

# search: cari pola PERTAMA, kembalikan Match object (atau None)
match = re.search(r'https?://\S+', text)
if match:
    print("search — URL pertama:", match.group())
    print("  Start:", match.start(), "End:", match.end())

# match: hanya cocok dari AWAL string
print("\nmatch dari awal:", re.match(r'https?://', text))  # None — teks tidak dimulai URL
print("match dari awal:", re.match(r'Check', text).group())

# fullmatch: keseluruhan string harus cocok
email = "user@example.com"
pattern_email = r'[\w.+-]+@[\w-]+\.[a-z]{2,}'
print("\nfullmatch email:", re.fullmatch(pattern_email, email))

# findall: SEMUA kemunculan sebagai list — paling sering dipakai
all_urls = re.findall(r'https?://\S+', text)
print("\nfindall semua URL:", all_urls)

search — URL pertama: https://huggingface.co/bert-base
  Start: 6 End: 38

match dari awal: None
match dari awal: Check

fullmatch email: <re.Match object; span=(0, 16), match='user@example.com'>

findall semua URL: ['https://huggingface.co/bert-base', 'http://arxiv.org/abs/1706.03762']


### 7. `re.sub` — Substitusi dan Pembersihan Teks
`re.sub` adalah fungsi regex yang **paling sering digunakan** dalam preprocessing dataset NLP/LLM. Seluruh pipeline pembersihan teks — menghapus URL, mention, hashtag, karakter khusus, spasi berlebih — hampir selalu dilakukan dengan `re.sub`. Menguasai ini berarti menguasai 80% kebutuhan text cleaning.

In [5]:
raw_tweet = "  @JohnDoe Hey!! Check out https://example.com/deep-learning #AI #PyTorch 🔥🚀 ... great model!!!  "

print("Original:", repr(raw_tweet))
print()

text = raw_tweet

# Hapus URL (http/https)
text = re.sub(r'https?://\S+|www\.\S+', '', text)
show("Hapus URL", text)

# Hapus @mention (Twitter/social media)
text = re.sub(r'@[A-Za-z0-9_]+', '', text)
show("Hapus @mention", text)

# Hapus #hashtag
text = re.sub(r'#\w+', '', text)
show("Hapus #hashtag", text)

# Hapus emoji dan karakter non-ASCII
text = re.sub(r'[^\x00-\x7F]+', '', text)
show("Hapus non-ASCII/emoji", text)

# Hapus karakter spesial (bukan huruf, angka, spasi, tanda baca dasar)
text = re.sub(r'[^a-zA-Z0-9\s.,!?\'\-]', '', text)
show("Hapus char spesial", text)

# Normalisasi tanda baca berulang (!!!! → !)
text = re.sub(r'([!?.,]){2,}', r'\1', text)
show("Normalisasi tanda baca", text)

# Normalisasi spasi berlebih + strip
text = re.sub(r'\s+', ' ', text).strip()
show("Normalisasi spasi", text)

Original: '  @JohnDoe Hey!! Check out https://example.com/deep-learning #AI #PyTorch 🔥🚀 ... great model!!!  '

  [Hapus URL] → '  @JohnDoe Hey!! Check out  #AI #PyTorch 🔥🚀 ... great model!!!  '
  [Hapus @mention] → '   Hey!! Check out  #AI #PyTorch 🔥🚀 ... great model!!!  '
  [Hapus #hashtag] → '   Hey!! Check out    🔥🚀 ... great model!!!  '
  [Hapus non-ASCII/emoji] → '   Hey!! Check out     ... great model!!!  '
  [Hapus char spesial] → '   Hey!! Check out     ... great model!!!  '
  [Normalisasi tanda baca] → '   Hey! Check out     . great model!  '
  [Normalisasi spasi] → 'Hey! Check out . great model!'


### 8. Compiled Pattern (`re.compile`) — Best Practice untuk Dataset Besar
Ketika sebuah pola digunakan berulang kali (misalnya membersihkan jutaan baris dataset), **kompilasi pola terlebih dahulu dengan `re.compile`** meningkatkan performa secara signifikan karena pola hanya di-parse sekali. Ini adalah best practice standar dalam pipeline preprocessing NLP skala besar.

In [6]:
import time

# Definisikan pola sekali, gunakan berulang kali
URL_PATTERN     = re.compile(r'https?://\S+|www\.\S+')
MENTION_PATTERN = re.compile(r'@[A-Za-z0-9_]+')
HASHTAG_PATTERN = re.compile(r'#\w+')
SPACES_PATTERN  = re.compile(r'\s+')
NON_ASCII       = re.compile(r'[^\x00-\x7F]+')

def clean_text(text: str) -> str:
    """Pipeline pembersihan teks standar untuk dataset NLP."""
    text = URL_PATTERN.sub('', text)
    text = MENTION_PATTERN.sub('', text)
    text = HASHTAG_PATTERN.sub('', text)
    text = NON_ASCII.sub('', text)
    text = SPACES_PATTERN.sub(' ', text).strip()
    return text.lower()

# Simulasi batch teks dari dataset
sample_texts = [
    "@elonmusk Check https://openai.com #GPT4 is amazing!! 🤖",
    "Paper: http://arxiv.org/abs/1706.03762 — Attention is all you need!",
    "Fine-tuning #BERT on @HuggingFace 🔥 transformers library",
    "Loss went from 2.5 to 0.8 after 3 epochs. See https://wandb.ai/run/abc",
]

print("Hasil pembersihan teks:")
for i, raw in enumerate(sample_texts):
    cleaned = clean_text(raw)
    print(f"  [{i}] Raw    : {raw!r}")
    print(f"       Cleaned : {cleaned!r}")
    print()

Hasil pembersihan teks:
  [0] Raw    : '@elonmusk Check https://openai.com #GPT4 is amazing!! 🤖'
       Cleaned : 'check is amazing!!'

  [1] Raw    : 'Paper: http://arxiv.org/abs/1706.03762 — Attention is all you need!'
       Cleaned : 'paper: attention is all you need!'

  [2] Raw    : 'Fine-tuning #BERT on @HuggingFace 🔥 transformers library'
       Cleaned : 'fine-tuning on transformers library'

  [3] Raw    : 'Loss went from 2.5 to 0.8 after 3 epochs. See https://wandb.ai/run/abc'
       Cleaned : 'loss went from 2.5 to 0.8 after 3 epochs. see'



### 9. Groups dan Capturing: Ekstraksi Informasi Terstruktur
Capturing groups `()` memungkinkan kita mengekstrak **bagian tertentu** dari pola yang cocok. Ini sangat berguna untuk mengurai format teks terstruktur seperti log training PyTorch/Hugging Face Trainer, konfigurasi model, atau metadata dataset — tanpa harus memisah string secara manual.

In [7]:
# --- Contoh 1: Parse log training PyTorch kustom ---
training_logs = [
    "Epoch 1/10 | Step 100/500 | Loss: 2.4532 | LR: 5e-05",
    "Epoch 3/10 | Step 300/500 | Loss: 1.2871 | LR: 3e-05",
    "Epoch 10/10 | Step 500/500 | Loss: 0.5412 | LR: 1e-06",
]

log_pattern = re.compile(
    r'Epoch (\d+)/(\d+) \| Step (\d+)/(\d+) \| Loss: ([\d.]+) \| LR: ([\de+-]+)'
)

print("=== Parse Log Training ===")
for log in training_logs:
    m = log_pattern.search(log)
    if m:
        epoch, total_epoch, step, total_step, loss, lr = m.groups()
        print(f"  Epoch {epoch}/{total_epoch} | Step {step}/{total_step} | loss={float(loss):.4f} | lr={lr}")

# --- Contoh 2: Ekstraksi nama model dari path HuggingFace-style ---
model_paths = [
    "./checkpoints/bert-base-uncased/epoch_3",
    "./checkpoints/gpt2-medium/epoch_10",
    "./checkpoints/distilbert-base/epoch_5",
]

path_pattern = re.compile(r'checkpoints/([\w-]+)/epoch_(\d+)')

print("\n=== Parse Checkpoint Path ===")
for path in model_paths:
    m = path_pattern.search(path)
    if m:
        model_name, epoch = m.group(1), m.group(2)
        print(f"  Model: {model_name} | Epoch: {epoch}")

# --- Contoh 3: Named groups (lebih readable) ---
metric_log = "val_loss=0.3412 val_acc=0.9231 val_f1=0.8975"
metric_pattern = re.compile(r'(?P<name>\w+)=(?P<value>[\d.]+)')

print("\n=== Named Groups — Ekstraksi Metrik ===")
for m in metric_pattern.finditer(metric_log):
    print(f"  {m.group('name'):12s} = {float(m.group('value')):.4f}")

=== Parse Log Training ===
  Epoch 1/10 | Step 100/500 | loss=2.4532 | lr=5e-05
  Epoch 3/10 | Step 300/500 | loss=1.2871 | lr=3e-05
  Epoch 10/10 | Step 500/500 | loss=0.5412 | lr=1e-06

=== Parse Checkpoint Path ===
  Model: bert-base-uncased | Epoch: 3
  Model: gpt2-medium | Epoch: 10
  Model: distilbert-base | Epoch: 5

=== Named Groups — Ekstraksi Metrik ===
  val_loss     = 0.3412
  val_acc      = 0.9231
  val_f1       = 0.8975


### 10. `re.finditer` — Iterasi Match dengan Posisi
`finditer` mengembalikan iterator of Match objects, bukan hanya list string. Ini penting ketika kita butuh **posisi** dari setiap kemunculan pola, misalnya untuk span-level annotation, NER (Named Entity Recognition) post-processing, atau menghighlight token tertentu dalam teks.

In [8]:
# Simulasi: teks dari dataset QA — cari semua angka dan posisinya
text = "The model was trained for 10 epochs with batch size 32 and learning rate 5e-5, achieving 92.4% accuracy on 15000 samples."

number_pattern = re.compile(r'\b[\d]+(?:[.]\d+)?(?:e[-+]?\d+)?\b')

print("Teks:", text)
print("\nSemua angka dengan posisi:")
for m in number_pattern.finditer(text):
    print(f"  Nilai: {m.group():10s} | Start: {m.start():3d} | End: {m.end():3d} | Span: {text[m.start():m.end()]!r}")

# --- Simulasi span annotation untuk NER output processing ---
ner_text = "Hugging Face released BERT in 2018 and GPT-2 was released by OpenAI."
org_pattern = re.compile(r'Hugging Face|OpenAI|Google|Meta|Microsoft', re.IGNORECASE)

print("\n=== Span Anotasi Organisasi ===")
annotations = []
for m in org_pattern.finditer(ner_text):
    annotations.append({'entity': 'ORG', 'word': m.group(), 'start': m.start(), 'end': m.end()})
    print(f"  ORG: {m.group()!r:15s} @ [{m.start()}, {m.end()}]")

print("\nAnnotations (format seperti HuggingFace NER output):", annotations)

Teks: The model was trained for 10 epochs with batch size 32 and learning rate 5e-5, achieving 92.4% accuracy on 15000 samples.

Semua angka dengan posisi:
  Nilai: 10         | Start:  26 | End:  28 | Span: '10'
  Nilai: 32         | Start:  52 | End:  54 | Span: '32'
  Nilai: 5e-5       | Start:  73 | End:  77 | Span: '5e-5'
  Nilai: 92.4       | Start:  89 | End:  93 | Span: '92.4'
  Nilai: 15000      | Start: 107 | End: 112 | Span: '15000'

=== Span Anotasi Organisasi ===
  ORG: 'Hugging Face'  @ [0, 12]
  ORG: 'OpenAI'        @ [61, 67]

Annotations (format seperti HuggingFace NER output): [{'entity': 'ORG', 'word': 'Hugging Face', 'start': 0, 'end': 12}, {'entity': 'ORG', 'word': 'OpenAI', 'start': 61, 'end': 67}]


### 11. Flags: `re.IGNORECASE`, `re.MULTILINE`, `re.DOTALL`
Flags mengubah perilaku pencocokan. `IGNORECASE` sangat umum untuk normalisasi teks. `MULTILINE` penting saat memproses teks multi-baris (seperti dokumen atau output model yang panjang). `DOTALL` (`.` cocok dengan newline) sering dibutuhkan saat parsing prompt atau template instruksi LLM.

In [9]:
# --- IGNORECASE: normalisasi tanpa peduli huruf besar/kecil ---
texts = ["BERT model", "bert model", "Bert Model", "bErT mOdEl"]
pattern = re.compile(r'bert', re.IGNORECASE)

print("=== IGNORECASE ===")
for t in texts:
    found = bool(pattern.search(t))
    print(f"  {t!r:20s} → ditemukan: {found}")

# --- MULTILINE: ^ dan $ cocok di setiap baris ---
log_block = """Epoch 1: loss=2.51
Epoch 2: loss=1.87
Epoch 3: loss=1.32
Epoch 4: loss=0.98"""

print("\n=== MULTILINE — Semua baris 'Epoch' ===")
epoch_lines = re.findall(r'^Epoch \d+: loss=[\d.]+$', log_block, re.MULTILINE)
for line in epoch_lines:
    print("  ", line)

# --- DOTALL: . cocok termasuk newline — untuk parse prompt multi-baris ---
prompt_template = """### Instruction:
Translate the following text.

### Input:
Hello world

### Response:
Halo dunia"""

print("\n=== DOTALL — Ekstrak blok Response ===")
response_match = re.search(r'### Response:\n(.+)', prompt_template, re.DOTALL)
if response_match:
    print("  Response:", repr(response_match.group(1).strip()))

=== IGNORECASE ===
  'BERT model'         → ditemukan: True
  'bert model'         → ditemukan: True
  'Bert Model'         → ditemukan: True
  'bErT mOdEl'         → ditemukan: True

=== MULTILINE — Semua baris 'Epoch' ===
   Epoch 1: loss=2.51
   Epoch 2: loss=1.87
   Epoch 3: loss=1.32
   Epoch 4: loss=0.98

=== DOTALL — Ekstrak blok Response ===
  Response: 'Halo dunia'


### 12. Lookahead dan Lookbehind (Non-Capturing Assertions)
Lookahead `(?=...)` dan lookbehind `(?<=...)` memungkinkan pencocokan berdasarkan konteks **tanpa menyertakan konteks tersebut** dalam hasil. Sangat berguna untuk memisahkan konten dari template prompt instruksi (format Alpaca, ChatML, dll.) yang umum digunakan saat fine-tuning LLM.

In [10]:
# --- Contoh 1: Lookahead — ambil angka yang diikuti '%' ---
stats_text = "Accuracy: 94.5% | F1: 88.3% | Loss: 0.312"

# Hanya ambil angka yang DIIKUTI '%' (tapi '%' tidak ikut dalam match)
percentages = re.findall(r'[\d.]+(?=%)', stats_text)
print("=== Lookahead — Angka sebelum '%' ===")
print("  Persentase:", percentages)

# --- Contoh 2: Lookbehind — ambil nilai setelah 'Loss: ' ---
loss_value = re.findall(r'(?<=Loss: )[\d.]+', stats_text)
print("\n=== Lookbehind — Nilai Loss ===")
print("  Loss value:", loss_value)

# --- Contoh 3: Parse format prompt Alpaca/Instruction-tuning ---
alpaca_sample = """Below is an instruction that describes a task.
### Instruction:
Summarize the following article in one sentence.

### Input:
Large language models have revolutionized NLP.

### Response:
LLMs have transformed the field of natural language processing."""

print("\n=== Parse Alpaca Prompt Format ===")

# Ekstrak tiap section dengan lookbehind
sections = {'Instruction': None, 'Input': None, 'Response': None}
for section in sections:
    pattern = rf'(?<=### {section}:\n)(.*?)(?=\n### |\Z)'
    m = re.search(pattern, alpaca_sample, re.DOTALL)
    if m:
        sections[section] = m.group(0).strip()

for key, val in sections.items():
    print(f"  {key}: {val!r}")

=== Lookahead — Angka sebelum '%' ===
  Persentase: ['94.5', '88.3']

=== Lookbehind — Nilai Loss ===
  Loss value: ['0.312']

=== Parse Alpaca Prompt Format ===
  Instruction: 'Summarize the following article in one sentence.'
  Input: 'Large language models have revolutionized NLP.'
  Response: 'LLMs have transformed the field of natural language processing.'


### 13. `re.split` — Tokenisasi Sederhana dan Pemisahan Teks
`re.split` memisahkan string berdasarkan pola regex, jauh lebih fleksibel dari `.split()` bawaan Python. Dalam NLP, ini digunakan untuk segmentasi kalimat sederhana (sebelum pakai `nltk`/`spacy`), memisahkan teks berdasarkan pola kompleks, atau memecah dokumen panjang menjadi chunk untuk LLM context window.

In [11]:
# --- Contoh 1: Segmentasi kalimat sederhana ---
paragraph = "The model achieved 94.5% accuracy. However, F1-score was lower at 88.3%! Should we retrain? Yes, with more data."

# Split pada '.', '?', '!' diikuti spasi
sentences = re.split(r'(?<=[.!?])\s+', paragraph)
print("=== Segmentasi Kalimat ===")
for i, s in enumerate(sentences):
    print(f"  [{i}] {s!r}")

# --- Contoh 2: Split dokumen panjang menjadi chunk (LLM chunking) ---
document = """Introduction: Large models have changed AI.
Section 1: Architecture overview of transformers.
Section 2: Training data and preprocessing steps.
Section 3: Fine-tuning strategies and RLHF.
Conclusion: Future of large language models."""

chunks = re.split(r'\n(?=\w)', document)  # split di newline yang diikuti huruf
print("\n=== Document Chunking ===")
for i, chunk in enumerate(chunks):
    print(f"  Chunk {i}: {chunk.strip()!r}")

# --- Contoh 3: Tokenisasi sederhana (pisahkan pada spasi & punctuation) ---
text = "Hello, world! This is PyTorch v2.0."
tokens = re.split(r'\s+|(?<=[\w])(?=[^\w\s])|(?<=[^\w\s])(?=[\w])', text)
tokens = [t for t in tokens if t]  # hapus string kosong
print("\n=== Tokenisasi Sederhana ===")
print("  Tokens:", tokens)

=== Segmentasi Kalimat ===
  [0] 'The model achieved 94.5% accuracy.'
  [1] 'However, F1-score was lower at 88.3%!'
  [2] 'Should we retrain?'
  [3] 'Yes, with more data.'

=== Document Chunking ===
  Chunk 0: 'Introduction: Large models have changed AI.'
  Chunk 1: 'Section 1: Architecture overview of transformers.'
  Chunk 2: 'Section 2: Training data and preprocessing steps.'
  Chunk 3: 'Section 3: Fine-tuning strategies and RLHF.'
  Chunk 4: 'Conclusion: Future of large language models.'

=== Tokenisasi Sederhana ===
  Tokens: ['Hello', ',', 'world', '!', 'This', 'is', 'PyTorch', 'v2', '.', '0', '.']


### 14. Pattern Penting untuk Preprocessing Teks NLP
Kumpulan pola regex yang paling sering ditemui dan digunakan dalam preprocessing dataset untuk model NLP/LLM. Dijadikan satu fungsi `full_clean_pipeline` yang bisa langsung dipakai atau dijadikan basis pipeline data cleaning produksi.

In [12]:
# ============================================================
# Kumpulan pattern penting — siap pakai
# ============================================================

PATTERNS = {
    'url'          : re.compile(r'https?://\S+|www\.\S+'),
    'mention'      : re.compile(r'@[A-Za-z0-9_]+'),
    'hashtag'      : re.compile(r'#\w+'),
    'email'        : re.compile(r'[\w.+-]+@[\w-]+\.[a-z]{2,}'),
    'html_tag'     : re.compile(r'<[^>]+>'),
    'number'       : re.compile(r'\b\d+(?:[.,]\d+)?\b'),
    'non_ascii'    : re.compile(r'[^\x00-\x7F]+'),
    'punct_repeat' : re.compile(r'([!?.]){2,}'),
    'multi_space'  : re.compile(r'\s+'),
    'newline'      : re.compile(r'[\r\n]+'),
    'special_char' : re.compile(r'[^a-zA-Z0-9\s.,!?\'\-]'),
    'leading_punct': re.compile(r'^[\s.,!?-]+|[\s.,!?-]+$'),
}

def full_clean_pipeline(
    text: str,
    remove_urls=True,
    remove_mentions=True,
    remove_hashtags=True,
    remove_html=True,
    remove_non_ascii=True,
    normalize_punct=True,
    lowercase=True
) -> str:
    """Pipeline pembersihan teks lengkap dengan kontrol flag per langkah."""
    if remove_html:       text = PATTERNS['html_tag'].sub(' ', text)
    if remove_urls:       text = PATTERNS['url'].sub(' ', text)
    if remove_mentions:   text = PATTERNS['mention'].sub(' ', text)
    if remove_hashtags:   text = PATTERNS['hashtag'].sub(' ', text)
    if remove_non_ascii:  text = PATTERNS['non_ascii'].sub('', text)
    if normalize_punct:   text = PATTERNS['punct_repeat'].sub(r'\1', text)
    text = PATTERNS['newline'].sub(' ', text)
    text = PATTERNS['multi_space'].sub(' ', text).strip()
    if lowercase:         text = text.lower()
    return text


# Test dengan beberapa contoh nyata
test_cases = [
    "<p>@GPT4 Check https://openai.com/gpt4 for details!!! 🤖 #AI</p>",
    "Training loss: 2.51 → 0.82\nEpoch 10/10 done! See results at http://wandb.ai",
    "user@hf.co sent: Fine-tune BERT with HuggingFace 🔥🔥🔥 ... works great!!!",
]

print("=== Full Clean Pipeline ===")
for raw in test_cases:
    cleaned = full_clean_pipeline(raw)
    print(f"  Raw     : {raw!r}")
    print(f"  Cleaned : {cleaned!r}")
    print()

=== Full Clean Pipeline ===
  Raw     : '<p>@GPT4 Check https://openai.com/gpt4 for details!!! 🤖 #AI</p>'
  Cleaned : 'check for details!'

  Raw     : 'Training loss: 2.51 → 0.82\nEpoch 10/10 done! See results at http://wandb.ai'
  Cleaned : 'training loss: 2.51 0.82 epoch 10/10 done! see results at'

  Raw     : 'user@hf.co sent: Fine-tune BERT with HuggingFace 🔥🔥🔥 ... works great!!!'
  Cleaned : 'user .co sent: fine-tune bert with huggingface . works great!'



### 15. Regex untuk Validasi dan Filter Data Training
Sebelum data masuk ke training loop, penting untuk memfilter sampel yang tidak berkualitas. Regex digunakan untuk mendeteksi dan menyaring teks yang terlalu pendek/panjang, terlalu banyak karakter non-huruf, mengandung konten tidak relevan (kode program, hanya angka, dll.).

In [13]:
import re
from typing import List, Dict

# Filter rules untuk kualitas data training
MOSTLY_PUNCT   = re.compile(r'^[\W\d\s]{10,}$')      # hampir semuanya bukan huruf
ONLY_NUMBERS   = re.compile(r'^[\d\s.,%-]+$')         # hanya angka dan tanda
REPEATED_WORD  = re.compile(r'\b(\w+)(\s+\1){3,}\b')  # kata yang diulang 3x+
CODE_BLOCK     = re.compile(r'(def |import |class |\{|}|<script)', re.IGNORECASE)
MIN_WORD_RATIO = re.compile(r'[a-zA-Z]')              # untuk hitung rasio huruf

def is_good_sample(text: str, min_words: int = 5, max_words: int = 512,
                   min_alpha_ratio: float = 0.5) -> Dict:
    """Validasi kualitas sampel teks untuk training dataset."""
    words = text.split()
    word_count = len(words)
    alpha_ratio = len(MIN_WORD_RATIO.findall(text)) / max(len(text), 1)

    issues = []
    if word_count < min_words:                 issues.append(f'too_short ({word_count} words)')
    if word_count > max_words:                 issues.append(f'too_long ({word_count} words)')
    if MOSTLY_PUNCT.match(text):               issues.append('mostly_punctuation')
    if ONLY_NUMBERS.match(text):               issues.append('only_numbers')
    if REPEATED_WORD.search(text):             issues.append('repeated_words')
    if CODE_BLOCK.search(text):                issues.append('contains_code')
    if alpha_ratio < min_alpha_ratio:          issues.append(f'low_alpha_ratio ({alpha_ratio:.2f})')

    return {'valid': len(issues) == 0, 'issues': issues, 'word_count': word_count}


test_samples = [
    "The quick brown fox jumps over the lazy dog, demonstrating natural language patterns.",  # valid
    "Hi",                                                                                       # too short
    "1234 5678 90.5% 42 100 200",                                                              # only numbers
    "good good good good good model",                                                           # repeated words
    "def forward(self): import torch\nclass MyModel(nn.Module):",                             # code
    "!!!...???...!!!...???...!!!",                                                              # mostly punct
]

print("=== Validasi Kualitas Sampel Data Training ===")
for sample in test_samples:
    result = is_good_sample(sample)
    status = '✓ VALID' if result['valid'] else '✗ INVALID'
    issues = result['issues'] if result['issues'] else ['-']
    print(f"  {status} | Words: {result['word_count']:3d} | Issues: {issues}")
    print(f"          Teks: {sample[:60]!r}")
    print()

=== Validasi Kualitas Sampel Data Training ===
  ✓ VALID | Words:  13 | Issues: ['-']
          Teks: 'The quick brown fox jumps over the lazy dog, demonstrating n'

  ✗ INVALID | Words:   1 | Issues: ['too_short (1 words)']
          Teks: 'Hi'

  ✗ INVALID | Words:   6 | Issues: ['mostly_punctuation', 'only_numbers', 'low_alpha_ratio (0.00)']
          Teks: '1234 5678 90.5% 42 100 200'

  ✗ INVALID | Words:   6 | Issues: ['repeated_words']
          Teks: 'good good good good good model'

  ✗ INVALID | Words:   6 | Issues: ['contains_code']
          Teks: 'def forward(self): import torch\nclass MyModel(nn.Module):'

  ✗ INVALID | Words:   1 | Issues: ['too_short (1 words)', 'mostly_punctuation', 'low_alpha_ratio (0.00)']
          Teks: '!!!...???...!!!...???...!!!'



### 16. Regex pada Instruksi Prompt dan Parsing Output LLM
Saat bekerja dengan LLM (GPT, LLaMA, Mistral, dsb.), sering kali output model perlu di-parse untuk mengekstrak bagian tertentu (jawaban, reasoning, kode), atau input prompt perlu diformat dan divalidasi menggunakan regex. Ini adalah keahlian inti untuk LLM application engineering.

In [14]:
# --- Contoh 1: Ekstrak blok kode dari output LLM ---
llm_output = """
Sure! Here is the Python code to train a simple model:

```python
import torch
model = torch.nn.Linear(10, 2)
optimizer = torch.optim.Adam(model.parameters(), lr=1e-3)
```

And here's a bash command to install dependencies:

```bash
pip install transformers datasets
```
"""

code_block_pattern = re.compile(r'```(\w+)?\n(.*?)```', re.DOTALL)

print("=== Ekstrak Code Block dari Output LLM ===")
for m in code_block_pattern.finditer(llm_output):
    lang = m.group(1) or 'unknown'
    code = m.group(2).strip()
    print(f"  Language: {lang}")
    print(f"  Code:\n{code}\n")

# --- Contoh 2: Parse output model dengan format JSON-like ---
structured_output = 'Answer: {"sentiment": "positive", "confidence": 0.94, "reason": "positive language used"}'

# Ekstrak pasangan key-value
kv_pattern = re.compile(r'"(\w+)":\s*("[^"]*"|[\d.]+)')
print("=== Parse Structured Output ===")
for m in kv_pattern.finditer(structured_output):
    key = m.group(1)
    val = m.group(2).strip('"')
    print(f"  {key}: {val}")

# --- Contoh 3: Validasi format prompt sebelum dikirim ke model ---
def validate_alpaca_prompt(prompt: str) -> bool:
    """Periksa apakah prompt mengikuti format Alpaca instruction-tuning."""
    has_instruction = bool(re.search(r'###\s*Instruction:', prompt))
    has_response    = bool(re.search(r'###\s*Response:', prompt))
    return has_instruction and has_response

prompts = [
    "### Instruction:\nSummarize this.\n\n### Response:\n",  # valid
    "Just answer the question directly.",                     # invalid
    "### instruction:\nDo something\n### Response:\n",        # valid (IGNORECASE needed)
]

print("\n=== Validasi Format Alpaca Prompt ===")
for p in prompts:
    valid = validate_alpaca_prompt(p)
    print(f"  {'✓' if valid else '✗'} {p[:55]!r}")

=== Ekstrak Code Block dari Output LLM ===
  Language: python
  Code:
import torch
model = torch.nn.Linear(10, 2)
optimizer = torch.optim.Adam(model.parameters(), lr=1e-3)

  Language: bash
  Code:
pip install transformers datasets

=== Parse Structured Output ===
  sentiment: positive
  confidence: 0.94
  reason: positive language used

=== Validasi Format Alpaca Prompt ===
  ✓ '### Instruction:\nSummarize this.\n\n### Response:\n'
  ✗ 'Just answer the question directly.'
  ✗ '### instruction:\nDo something\n### Response:\n'
